In [1]:
import pandas as pd

flights = pd.read_csv("T_ONTIME_REPORTING.csv")
flights.head()

,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DEL15,CANCELLED,CANCELLATION_CODE
0,1/1/2026 12:00:00 AM,AA,1,12478,1247805,31703,JFK,12892,1289209,32575,LAX,700,657.0,-3.0,0.0,0.0,NaN
1,1/1/2026 12:00:00 AM,AA,10,12892,1289209,32575,LAX,12478,1247805,31703,JFK,2130,2128.0,-2.0,0.0,0.0,NaN
2,1/1/2026 12:00:00 AM,AA,1002,13303,1330304,32467,MIA,13495,1349506,33495,MSY,2245,2252.0,7.0,0.0,0.0,NaN
3,1/1/2026 12:00:00 AM,AA,1003,11292,1129203,30325,DEN,13303,1330304,32467,MIA,2338,2338.0,0.0,0.0,0.0,NaN
4,1/1/2026 12:00:00 AM,AA,1004,10721,1072103,30721,BOS,11057,1105704,31057,CLT,757,756.0,-1.0,0.0,0.0,NaN


In [2]:
clean_flights = flights.copy()
clean_flights["ORIGIN"] = clean_flights["ORIGIN"].str.strip().str.upper()
clean_flights["FL_DATE"] = pd.to_datetime(
    clean_flights["FL_DATE"], format="mixed", errors="coerce"
).dt.normalize()

clean_flights = clean_flights.loc[
    clean_flights["ORIGIN"].eq("ORD")
    & clean_flights["FL_DATE"].between("2026-01-01", "2026-01-31")
]

before = len(clean_flights)
clean_flights = clean_flights.drop_duplicates()

print(f"O'Hare rows before cleaning: {before:,}")
print(f"Duplicate rows removed: {before - len(clean_flights):,}")
print(f"O'Hare rows saved: {len(clean_flights):,}")

clean_flights.to_csv("flights_ord_jan2026_clean.csv", index=False)

O'Hare rows before cleaning: 25,131
Duplicate rows removed: 0
O'Hare rows saved: 25,131


In [3]:
airport_columns = [
    "airport_id", "name", "city", "country", "iata", "icao",
    "latitude", "longitude", "altitude_ft", "utc_offset", "dst",
    "timezone", "type", "source"
]

airports = pd.read_csv(
    "airports.dat.txt",
    header=None,
    names=airport_columns,
    na_values=r"\N"
)

airports["iata"] = airports["iata"].str.strip().str.upper()

clean_airports = airports.loc[
    airports["iata"].eq("ORD"),
    ["iata", "icao", "name", "latitude", "longitude", "timezone"]
].drop_duplicates()

assert len(clean_airports) == 1, "Expected one airport record for ORD."

clean_airports.to_csv("airports_ord_clean.csv", index=False)
display(clean_airports)

,iata,icao,name,latitude,longitude,timezone
3630,ORD,KORD,Chicago O'Hare International Airport,41.9786,-87.9048,America/Chicago


In [5]:
weather = pd.read_csv("LCD_USW00094846_2026.csv", low_memory=False)
weather["DATE"] = pd.to_datetime(weather["DATE"], errors="coerce")

weather_fields = [
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    "HourlyWindSpeed",
    "HourlyVisibility",
]
in_january = (weather["DATE"] >= "2026-01-01") & (weather["DATE"] < "2026-02-01")
has_reading = weather[weather_fields].notna().any(axis=1)
columns = ["STATION", "DATE", "REPORT_TYPE", "SOURCE"] + weather_fields

clean_weather = weather.loc[in_january & has_reading, columns].copy()
before = len(clean_weather)
clean_weather = clean_weather.drop_duplicates().sort_values("DATE")
clean_weather.insert(0, "iata", "ORD")
print(f"January weather rows before deduplication: {before:,}")
print(f"Duplicate rows removed: {before - len(clean_weather):,}")
print(f"Weather rows saved: {len(clean_weather):,}")
display(clean_weather[weather_fields].isna().sum().rename("Missing readings"))
clean_weather.to_csv("weather_ord_jan2026_clean.csv", index=False)

January weather rows before deduplication: 1,311
Duplicate rows removed: 0
Weather rows saved: 1,311


HourlyDryBulbTemperature      0
HourlyPrecipitation         883
HourlyWindSpeed               0
HourlyVisibility              0
Name: Missing readings, dtype: int64

In [ ]:
flight_fields = ["FL_DATE", "ORIGIN", "CRS_DEP_TIME", "CANCELLED", "DEP_DELAY"]
print("Missing values in flight fields:")
display(clean_flights[flight_fields].isna().sum())
cancelled = clean_flights["CANCELLED"].eq(1)
no_delay = clean_flights["DEP_DELAY"].isna()

print("Canceled flights with no delay recorded:", (cancelled & no_delay).sum())
print("Non canceled flights with no delay recorded:",
      (clean_flights["CANCELLED"].eq(0) & no_delay).sum())
print("Weather rows sharing a station and timestamp:",
      clean_weather.duplicated(["STATION", "DATE"], keep=False).sum())

Missing values in flight fields:


FL_DATE           0
ORIGIN            0
CRS_DEP_TIME      0
CANCELLED         0
DEP_DELAY       909
dtype: int64

Canceled flights with no delay recorded: 909
Non-canceled flights with no delay recorded: 0
Weather rows sharing a station and timestamp: 0
